# Using MCP Tools with LangChain

In this notebook, we will explore how to create a reasoning action agent using tools exposed by an MCP server with LangChain.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/ai-agents/02-a-mcp-tools.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Launch an MCP server and connect to it from a Python client.
- Create a ReAct agent using LangChain with MCP tools.
- Understand how tool binding, tool calling, and result passing work step by step.

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [1]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/rag/aup_config.py

In [2]:
from aup_config import aup_setup
aup_setup()

## Enhance the LLM with Tools

Use `ChatOpenAI` to connect to the Ollama local endpoint using its OpenAI-compatible API

In [3]:
!pip install langchain-ollama
!pip install mcp
!pip install mcp langchain-mcp-adapters
from langchain_ollama import ChatOllama

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

In [4]:
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    
)

### Test the connection

In [5]:
response = llm.invoke("Hello, introduce yourself")

print(response.content)

I'm happy to meet you. I am an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


### Create a Medicine Tool.

In [6]:
from langchain_core.tools import tool

@tool
def medicine_information(medicine_name: str):
    """
    Provides basic medicine information.
    """

    medicines = {
        "paracetamol": "Used for fever and mild pain relief.",
        "ibuprofen": "Used for pain and inflammation."
    }

    return medicines.get(
        medicine_name.lower(),
        "Medicine information not available."
    )

In [7]:
print(medicine_information)

name='medicine_information' description='Provides basic medicine information.' args_schema=<class 'langchain_core.utils.pydantic.medicine_information'> func=<function medicine_information at 0x7781c5be8860>


### Tools list

In [8]:
tools = [medicine_information]

print(tools)

[StructuredTool(name='medicine_information', description='Provides basic medicine information.', args_schema=<class 'langchain_core.utils.pydantic.medicine_information'>, func=<function medicine_information at 0x7781c5be8860>)]


### Bind tool

In [9]:
llm_with_tools = llm.bind_tools(tools)

### Test

In [10]:
response = llm_with_tools.invoke(
    "What is Paracetamol used for?"
)

print(response)

content='' additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-07-22T17:19:49.545735921Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3018880360, 'load_duration': 177360145, 'prompt_eval_count': 160, 'prompt_eval_duration': 1614479000, 'eval_count': 21, 'eval_duration': 1222745000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'} id='lc_run--019f8ad7-315d-7523-ab83-9c66591ceea9-0' tool_calls=[{'name': 'medicine_information', 'args': {'medicine_name': 'Paracetamol'}, 'id': '9afb6227-c147-438e-bd3f-e6ed5c4d8038', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 160, 'output_tokens': 21, 'total_tokens': 181}


### Execute the tool

In [11]:
tool_call = response.tool_calls[0]

tool_result = medicine_information.invoke(
    tool_call["args"]
)

print(tool_result)

Used for fever and mild pain relief.


### Create MCP Server

Create a new file:
      medicine_server.py

In [ ]:
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("Medicine Server")


@mcp.tool()
def get_medication_info(medicine_name: str) -> str:
    """
    Query medicine database for dosage, purpose and warnings.
    """

    db = {
        "metformin": "Metformin: Take 1 tablet (500mg) twice daily with meals. Purpose: Type 2 Diabetes management. Warning: Do not skip meals.",
        
        "lisinopril": "Lisinopril: Take 1 tablet (10mg) in the morning. Purpose: Blood Pressure management. Warning: Monitor BP regularly.",
        
        "aspirin": "Aspirin: Take 1 low-dose tablet (81mg) daily. Warning: Increased bleeding risk."
    }

    return db.get(
        medicine_name.lower().strip(),
        f"The medicine '{medicine_name}' was not found. Consult your doctor."
    )


await mcp.run_sse_async()

INFO:     Started server process [99]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [ ]:
from mcp.client.sse import sse_client
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools


async with sse_client("http://127.0.0.1:8000/sse") as (read, write):

    async with ClientSession(read, write) as session:

        await session.initialize()

        tools = await session.list_tools()

        print(tools)

Then connect these tools to your LangGraph agent:

In [ ]:
model_with_tools = model.bind_tools(mcp_tools)

Then add edges:

In [ ]:
graph.add_edge("tools", END)

and use:

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(mcp_tools)

### Start MCP Session in Jupyter

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

Create server parameters:

In [ ]:
server_params = StdioServerParameters(
    command="python",
    args=["medicine_server.py"]
)

### Connect MCP Client

In [ ]:
from mcp.client.sse import sse_client
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools


async with sse_client("http://127.0.0.1:8000/sse") as (read, write):

    async with ClientSession(read, write) as session:

        await session.initialize()

        mcp_tools = await load_mcp_tools(session)

        print(mcp_tools)

### Create MCP Client



In [ ]:
from mcp.client.sse import sse_client
from mcp import ClientSession
from langchain_mcp_adapters.tools import load_mcp_tools


async with stdio_client(server_params) as (read, write):

    async with ClientSession(read, write) as session:

        await session.initialize()

        mcp_tools = await load_mcp_tools(session)

        print(mcp_tools)

### Bind MCP tools to Llama

Your previous variable:

In [ ]:
llm

Use:

In [ ]:
model_with_medical_tools = llm.bind_tools(mcp_tools)

### Give Medical System Prompt

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


system_prompt = SystemMessage(content="""
You are the AI Medicine Reminder & Health Assistant.

Rules:
1. Give structured medicine information.
2. Never invent dosage information.
3. If medicine is unavailable, advise the user to contact a doctor.
""")

### Ask Medicine Question

In [ ]:
messages = [
    system_prompt,
    HumanMessage(
        content="What is the standard dosage for Lisinopril?"
    )
]


ai_response = await model_with_medical_tools.ainvoke(messages)

### Check Tool Decision

In [ ]:
print("--- Tool Calls ---")

for tool_call in ai_response.tool_calls:
    print("Tool:", tool_call["name"])
    print("Arguments:", tool_call["args"])

### Execute MCP Tool

In [ ]:
get_medication_info.invoke()

Now MCP uses:

In [ ]:
result = await session.call_tool(
    tool_call["name"],
    tool_call["args"]
)

### Send Result Back to Llama

In [ ]:
from langchain_core.messages import ToolMessage


tool_message = ToolMessage(
    content=str(result),
    tool_call_id=tool_call["id"],
    name=tool_call["name"]
)


final_messages = messages + [
    ai_response,
    tool_message
]


final_response = await model_with_medical_tools.ainvoke(
    final_messages
)


print(final_response.content)

## Exercises for the Reader

- Add your own MCP tool to `math_server.py` (for example, a `power` tool) and test it with the agent.
- Make one of the MCP tools return an incorrect value. What happens?

## Conclusions

In this notebook we launched an MCP server using `FastMCP` and connected to it via `stdio`. We then loaded the MCP tools and used them with a LangChain ReAct agent that reasons about queries and calls the appropriate tool. Finally, we broke down the tool-calling process into its three stages: binding tool schemas to the LLM, receiving structured tool calls from the LLM's response, and executing the tools on the client side before passing results back to the LLM for a final answer.

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://modelcontextprotocol.io/docs/getting-started/intro">Model Context Protocol</a></li>
</ul>
</div>

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT